# DeepMeow — Google Colab Pipeline

This notebook is the **team's shared entry point** for running DeepMeow on Google Colab.

**Workflow:**
1. Clone repo from GitHub
2. Install dependencies
3. Mount Google Drive to persist data
4. Download COCO cat dataset
5. Verify backbone forward pass

> **Tip:** Go to `Runtime -> Change runtime type -> T4 GPU` before running!

## Step 1 — Verify GPU

In [1]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU available: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
else:
    print('WARNING: No GPU found! Go to Runtime -> Change runtime type -> T4 GPU')

GPU available: Tesla T4 (15.6 GB VRAM)


## Step 2 — Clone Repository & Install Dependencies

In [2]:
import os

REPO_URL = 'https://github.com/IliyaJz/DeepMeow.git'
REPO_DIR = '/content/DeepMeow'

# Clone the repo (or pull latest changes if already cloned)
if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned. Pulling latest changes...')
    !git -C {REPO_DIR} pull

# Move into the repo directory — all relative paths in our code work from here
os.chdir(REPO_DIR)
print(f'\nWorking directory: {os.getcwd()}')

Cloning repository...
Cloning into '/content/DeepMeow'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 37 (delta 13), reused 37 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 24.38 KiB | 8.13 MiB/s, done.
Resolving deltas: 100% (13/13), done.

Working directory: /content/DeepMeow


In [3]:
# Install all required Python packages
print('Installing dependencies...')
!pip install -q -r requirements.txt
print('All packages installed!')

Installing dependencies...
All packages installed!


## Step 3 — Mount Google Drive

**Why?** Colab resets its disk every session (~12 hours). When Google Drive is mounted, the downloaded dataset (~500 MB) is saved there and you **won't need to re-download it next session**.

- **Team use**: Each member mounts their own Drive but clones the same code from GitHub.

In [4]:
USE_DRIVE = True  # Saved to Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted!')
    print('Data will be saved to Google Drive.')
else:
    print('Skipping Drive mount. Data will be stored in Colab /content/ (resets each session).')

Mounted at /content/drive
Google Drive mounted!
Data will be saved to Google Drive.


## Step 4 — Download the COCO Cat Dataset

This downloads:
- **~241 MB** COCO 2017 annotation file (JSON)
- **~3,000 train + 500 val** cat images (selected from the 80-class COCO dataset)

On Colab's fast connection this takes about **3–5 minutes** total.

In [5]:
!python src/data/downloader.py

 DeepMeow Dataset Downloader

Downloading: annotations_trainval2017.zip
  Progress: 100% 253M/253M [00:04<00:00, 53.4MB/s]
  Saved to data/tmp/annotations_trainval2017.zip

Extracting annotations ZIP...
  Extracted.

Filtering COCO 'train' annotations for cats...
  3000 images, 3444 annotations
  Saved -> data/annotations/train.json

  train: 100% 3000/3000 [19:43<00:00,  2.53it/s]
  Images saved to: data/raw/train

Filtering COCO 'val' annotations for cats...
  184 images, 202 annotations
  Saved -> data/annotations/val.json

  val: 100% 184/184 [01:10<00:00,  2.62it/s]
  Images saved to: data/raw/val

Cleaning up temporary files...
  Done!

 Dataset ready!
    Train images : data/raw/train/
    Val images   : data/raw/val/
    Annotations  : data/annotations/train.json & val.json


## Step 5 — Verify Dataset

In [6]:
import json
from pathlib import Path

# Determine if data was saved in Drive or local repo
data_root = Path('/content/drive/MyDrive/DeepMeow/data') if USE_DRIVE else Path('data')

for split in ['train', 'val']:
    ann_path   = data_root / f'annotations/{split}.json'
    image_dir  = data_root / f'raw/{split}'

    if ann_path.exists():
        with open(ann_path) as f:
            ann = json.load(f)

        n_images      = len(ann['images'])
        n_annotations = len(ann['annotations'])
        n_files       = len(list(image_dir.glob('*.jpg')))

        print(f'{split.upper()}:')
        print(f'  Annotation images : {n_images}')
        print(f'  Annotation boxes  : {n_annotations}')
        print(f'  Downloaded files  : {n_files} .jpg files')
        print()
    else:
        print(f'{split.upper()}: Annotations file not found at {ann_path}')

TRAIN: Annotations file not found at /content/drive/MyDrive/DeepMeow/data/annotations/train.json
VAL: Annotations file not found at /content/drive/MyDrive/DeepMeow/data/annotations/val.json


## Step 6 — Visualize Sample Images with Bounding Boxes

In [7]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

data_root = Path('/content/drive/MyDrive/DeepMeow/data') if USE_DRIVE else Path('data')
ann_file  = data_root / 'annotations/train.json'
train_dir = data_root / 'raw/train'

if ann_file.exists():
    with open(ann_file) as f:
        ann_data = json.load(f)

    id_to_anns = {}
    for ann in ann_data['annotations']:
        id_to_anns.setdefault(ann['image_id'], []).append(ann)

    id_to_img = {img['id']: img for img in ann_data['images']}

    valid_ids = [
        img_id for img_id in id_to_anns
        if (train_dir / id_to_img[img_id]['file_name']).exists()
    ]
    sample_ids = random.sample(valid_ids, min(6, len(valid_ids)))

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Sample Training Images with Ground-Truth Cat Boxes', fontsize=14)

    for ax, img_id in zip(axes.flat, sample_ids):
        img_info = id_to_img[img_id]
        img_path = train_dir / img_info['file_name']

        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"ID: {img_id} | {img_info['width']}x{img_info['height']}", fontsize=9)

        for ann in id_to_anns.get(img_id, []):
            x, y, w, h = ann['bbox']
            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2, edgecolor='#FF6B35', facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x, y - 4, 'cat', color='#FF6B35', fontsize=8, fontweight='bold')

    os.makedirs('results', exist_ok=True)
    plt.tight_layout()
    plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Sample visualization saved to results/sample_images.png')

## Step 7 — Verify CNN Backbone (Forward Pass Test)

In [8]:
import torch
from src.models.backbone import Backbone

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Build the backbone and move it to GPU
backbone = Backbone().to(device)

# Count parameters
total_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f'Backbone parameters: {total_params:,}')

# Run a fake batch through the backbone
dummy = torch.randn(2, 3, 416, 416, device=device)

with torch.no_grad():
    p3, p4, p5 = backbone(dummy)

print(f'\nFeature map shapes:')
print(f'  P3 (small cats) : {tuple(p3.shape)}   <- 52x52 grid')
print(f'  P4 (medium cats): {tuple(p4.shape)}  <- 26x26 grid')
print(f'  P5 (large cats) : {tuple(p5.shape)}  <- 13x13 grid')
print('\nBackbone forward pass OK!')

Using device: cuda
Backbone parameters: 40,584,928

Feature map shapes:
  P3 (small cats) : (2, 256, 52, 52)   <- 52x52 grid
  P4 (medium cats): (2, 512, 26, 26)  <- 26x26 grid
  P5 (large cats) : (2, 1024, 13, 13)  <- 13x13 grid

Backbone forward pass OK!


## Setup Complete!

Your environment is ready. Here's what you have:

| Component | Status |
|-----------|--------|
| Repository | Cloned from GitHub |
| Dependencies | Installed |
| Google Drive | Mounted |
| Dataset | Downloaded |
| Backbone | Verified |

**Next**: The team will implement the **FPN Neck + Detection Head** (Week 2). Pull the latest changes with `!git pull` to get new code as it's pushed to GitHub.